# 01 — Audit awal data SO₂

Tujuan: memeriksa struktur file sebelum mengimpor tabel untuk analisis.
File Excel hanya dibaca, tidak disimpan ulang. Tidak ada penghapusan baris,
imputasi, perubahan outlier, scaling, atau pelatihan model.

Jalankan seluruh sel dengan kernel `.venv` proyek. Nomor baris pada keluaran
mengikuti Excel (mulai 1). Sel kosong dipertahankan sebagai `None` pada pembacaan
openpyxl dan ditampilkan sebagai nilai kosong; angka nol tetap nol.
Nama parameter dan satuan disalin dari file, tanpa menafsirkan kode.

In [1]:
from pathlib import Path
from itertools import islice
from datetime import datetime
from hashlib import sha256
from importlib.metadata import version
import sys, json
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import openpyxl
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'data_raw').is_dir() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'data_raw').is_dir(), 'Jalankan dari root proyek atau folder notebooks.'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Python:', sys.version)
print('Executable:', sys.executable)
display(pd.DataFrame([{'Paket': p, 'Versi': version(p)} for p in
    ['pandas', 'numpy', 'matplotlib', 'seaborn', 'openpyxl', 'ipykernel']]))

def digest(path):
    with path.open('rb') as stream:
        return sha256(stream.read()).hexdigest()

files = sorted(p for p in (ROOT / 'data_raw').rglob('*')
    if p.is_file() and p.suffix.lower() in {'.xlsx', '.xlsm', '.xls', '.xlsb', '.xltx', '.xltm'}
    and not p.name.startswith('~$'))
before = {str(p.relative_to(ROOT)): digest(p) for p in files}
display(pd.DataFrame([{'File': str(p.relative_to(ROOT)), 'Byte': p.stat().st_size,
    'SHA256': before[str(p.relative_to(ROOT))]} for p in files]))
if not files:
    print('KENDALA: belum ada file Excel di data_raw.')

Python: 3.15.2 (tags/v3.15.2:5e58eaa, Oct 25 2026, 17:27:03) [MSC v.1951 64 bit (AMD64)]
Executable: C:\Users\INTAN\airpollutan_SO2\.venv\Scripts\python.exe


Matplotlib is building the font cache; this may take a moment.


,Paket,Versi
0,pandas,3.1.1
1,numpy,2.5.4
2,matplotlib,3.11.2
3,seaborn,0.13.2
4,openpyxl,3.1.5
5,ipykernel,7.4.0


,File,Byte,SHA256
0,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,17594455,bd00734b954e8df024c048575fad07f1eb1cbe1083d4c8929ae362aeb45a7406


## Pembacaan struktur tanpa header otomatis

Pratinjau menampilkan 20 baris pertama beserta posisi sel Excel. Pembacaan memakai
`read_only=True, data_only=False`, sehingga rumus tetap terlihat sebagai rumus.
Untuk file yang sudah diperiksa, nama kolom berada di baris 8 dan satuan di baris 9.
Notebook memvalidasi pola ini pada setiap sheet; layout lain dilaporkan untuk
pemeriksaan manual, bukan dipaksakan menjadi tabel. Lima baris awal berarti lima
baris bertimestamp pertama, termasuk bila seluruh nilai pengukurannya kosong.
Awal nilai pengukuran dan awal nilai SO2 dicatat terpisah dari awal timestamp.

In [2]:
summary, issues, details, inventories = [], [], [], []

def is_timestamp(value):
    if isinstance(value, datetime):
        return True
    if isinstance(value, str):
        try:
            datetime.strptime(value, '%Y-%m-%d %H:%M:%S')
            return True
        except ValueError:
            pass
    return False

for path in files:
    relative = str(path.relative_to(ROOT))
    display(Markdown(f'## File: {path.name}'))
    if path.suffix.lower() in {'.xls', '.xlsb'}:
        message = f'{relative}: format {path.suffix} membutuhkan engine tambahan; belum dibaca.'
        issues.append(message)
        print('KENDALA:', message)
        continue
    try:
        wb = openpyxl.load_workbook(path, read_only=True, data_only=False)
    except Exception as exc:
        issues.append(f'{relative}: {type(exc).__name__}: {exc}')
        print('KENDALA:', issues[-1])
        continue
    try:
        print('Seluruh sheet:', wb.sheetnames)
        inventories.append({'file': relative, 'sheets': wb.sheetnames})
        for ws in wb.worksheets:
            display(Markdown(f'### Sheet: {ws.title}'))
            preview = list(ws.iter_rows(min_row=1, max_row=20, values_only=True))
            letters = [openpyxl.utils.get_column_letter(i + 1) for i in range(ws.max_column)]
            display(pd.DataFrame(preview, columns=letters, index=range(1, len(preview)+1), dtype=object)
                    .rename_axis('Baris Excel'))
            # Validasi terhadap struktur yang telah ditinjau, bukan asumsi header baris pertama.
            header = list(preview[7]) if len(preview) >= 10 else []
            units = list(preview[8]) if len(preview) >= 10 else []
            if len(header) < 15 or header[1] != 'Waktu' or header[4] != 'SO2' or units[4] != 'µg/m³' or not is_timestamp(preview[9][1]):
                message = f'{relative} / {ws.title}: layout berbeda; tentukan header, satuan, dan awal data secara manual.'
                issues.append(message)
                print('KENDALA:', message)
                continue
            column_map = [{'Kolom Excel': letters[i], 'Nama pada file': header[i],
                           'Satuan pada file': units[i]} for i in range(1,15)]
            print('Header: baris 8; satuan: baris 9; awal baris bertimestamp: 10; tabel: B:O.')
            print('Satuan Waktu kosong; zona waktu tidak dijelaskan pada header.')
            display(pd.DataFrame(column_map, dtype=object))
            first_five = []
            count = 0
            first_time = last_time = first_measurement = first_so2 = None
            formula_count = 0
            other_nonempty = []
            for row_number, row in enumerate(ws.iter_rows(min_row=10, values_only=True), start=10):
                formula_count += sum(isinstance(v, str) and v.startswith('=') for v in row)
                if len(row) > 1 and is_timestamp(row[1]):
                    count += 1
                    first_time = row[1] if first_time is None else first_time
                    last_time = row[1]
                    if len(first_five) < 5:
                        first_five.append({'Baris Excel': row_number, **{header[i]: row[i] for i in range(1,15)}})
                    if first_measurement is None and any(v is not None for v in row[2:15]):
                        first_measurement = {'baris': row_number, 'waktu': row[1]}
                    if first_so2 is None and row[4] is not None:
                        first_so2 = {'baris': row_number, 'waktu': row[1], 'nilai': row[4]}
                elif any(v is not None for v in row):
                    other_nonempty.append(row_number)
            print('Lima baris observasi awal (nilai asli, tanpa pengisian sel kosong):')
            display(pd.DataFrame(first_five, dtype=object).set_index('Baris Excel'))
            record = {'File': relative, 'Sheet': ws.title, 'Header': 8, 'Satuan': 9,
                      'Awal timestamp': 10, 'Jumlah baris bertimestamp': count,
                      'Timestamp baris pertama': first_time, 'Timestamp baris terakhir': last_time,
                      'Awal nilai pengukuran (baris)': first_measurement['baris'] if first_measurement else None,
                      'Awal SO2 (baris)': first_so2['baris'] if first_so2 else None,
                      'Rumus setelah header': formula_count,
                      'Baris terpakai lain setelah header': len(other_nonempty)}
            summary.append(record)
            details.append({'file': relative, 'sheet': ws.title, 'kolom': column_map,
                            'metadata_baris_1_7': preview[:7], 'lima_observasi': first_five,
                            'awal_pengukuran': first_measurement, 'awal_so2': first_so2,
                            'baris_lain': other_nonempty})
            display(pd.DataFrame([record]))
    finally:
        wb.close()

Seluruh sheet: ['Bandar Lampung Talang', 'Kab. Lampung Selatan Way Urang', 'Kota Cilegon Pulomerak', 'Kab. Serang Binuang', 'Kab. Tangerang Curug', 'Tangerang Pasir Jaya', 'Tangerang Selatan Serpong', 'Serang Sumurpecung']
Header: baris 8; satuan: baris 9; awal baris bertimestamp: 10; tabel: B:O.
Satuan Waktu kosong; zona waktu tidak dijelaskan pada header.
Lima baris observasi awal (nilai asli, tanpa pengisian sel kosong):
Header: baris 8; satuan: baris 9; awal baris bertimestamp: 10; tabel: B:O.
Satuan Waktu kosong; zona waktu tidak dijelaskan pada header.
Lima baris observasi awal (nilai asli, tanpa pengisian sel kosong):
Header: baris 8; satuan: baris 9; awal baris bertimestamp: 10; tabel: B:O.
Satuan Waktu kosong; zona waktu tidak dijelaskan pada header.
Lima baris observasi awal (nilai asli, tanpa pengisian sel kosong):
Header: baris 8; satuan: baris 9; awal baris bertimestamp: 10; tabel: B:O.
Satuan Waktu kosong; zona waktu tidak dijelaskan pada header.
Lima baris observasi awal

## File: Telkom University Purwokerto - Intan Azizah.xlsx

### Sheet: Bandar Lampung Talang

,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O
Baris Excel,,,,,,,,,,,,,,,
1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,None,None,None,LAPORAN KONSENTRASI DAN METEOROLOGI,None,None,None,None,None,None,None,None,None,None,None
3,None,None,None,Data Monitoring di Stasiun Pemantau,None,None,None,None,None,None,None,None,None,None,None
4,None,None,None,Tanggal,: 01 Januari 2025 - 22 September 2026,None,None,None,None,None,None,None,None,None,None
5,None,None,None,Kota,: Bandar Lampung,None,None,None,None,None,None,None,None,None,None
6,None,None,None,Stasiun,: Bandar Lampung Talang,None,None,None,None,None,None,None,None,None,None
7,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
8,None,Waktu,PM10,PM2.5,SO2,CO,O3,NO2,Kec.Angin,Arah Angin,Kelembaban,Suhu,Tek.Udara,Sol.Rad,Curah Hujan
9,None,None,µg/m³,µg/m³,µg/m³,µg/m³,µg/m³,µg/m³,m/s,°,%,°C,mBar,watt/m²,mm


,Kolom Excel,Nama pada file,Satuan pada file
0,B,Waktu,None
1,C,PM10,µg/m³
2,D,PM2.5,µg/m³
3,E,SO2,µg/m³
4,F,CO,µg/m³
5,G,O3,µg/m³
6,H,NO2,µg/m³
7,I,Kec.Angin,m/s
8,J,Arah Angin,°
9,K,Kelembaban,%


,Waktu,PM10,PM2.5,SO2,CO,O3,NO2,Kec.Angin,Arah Angin,Kelembaban,Suhu,Tek.Udara,Sol.Rad,Curah Hujan
Baris Excel,,,,,,,,,,,,,,
10,2025-01-01 00:00:00,None,None,68,1955,None,59,0,0,0,0,0,0,0
11,2025-01-01 00:30:00,None,1,70,2003,None,59,0,0,0,0,0,0,0
12,2025-01-01 01:00:00,None,29,70,1999,None,59,0,0,0,0,0,0,0
13,2025-01-01 01:30:00,None,22,71,2089,None,60,0,0,0,0,0,0,0
14,2025-01-01 02:00:00,None,None,71,2154,None,61,0,0,0,0,0,0,0


... (32 outputs left out)

,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O
Baris Excel,,,,,,,,,,,,,,,
1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,None,None,None,LAPORAN KONSENTRASI DAN METEOROLOGI,None,None,None,None,None,None,None,None,None,None,None
3,None,None,None,Data Monitoring di Stasiun Pemantau,None,None,None,None,None,None,None,None,None,None,None
4,None,None,None,Tanggal,: 01 Januari 2025 - 19 September 2026,None,None,None,None,None,None,None,None,None,None
5,None,None,None,Kota,: Serang,None,None,None,None,None,None,None,None,None,None
6,None,None,None,Stasiun,: Serang Sumurpecung,None,None,None,None,None,None,None,None,None,None
7,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
8,None,Waktu,PM10,PM2.5,SO2,CO,O3,NO2,Kec.Angin,Arah Angin,Kelembaban,Suhu,Tek.Udara,Sol.Rad,Curah Hujan
9,None,None,µg/m³,µg/m³,µg/m³,µg/m³,µg/m³,µg/m³,m/s,°,%,°C,mBar,watt/m²,mm


,Kolom Excel,Nama pada file,Satuan pada file
0,B,Waktu,None
1,C,PM10,µg/m³
2,D,PM2.5,µg/m³
3,E,SO2,µg/m³
4,F,CO,µg/m³
5,G,O3,µg/m³
6,H,NO2,µg/m³
7,I,Kec.Angin,m/s
8,J,Arah Angin,°
9,K,Kelembaban,%


,Waktu,PM10,PM2.5,SO2,CO,O3,NO2,Kec.Angin,Arah Angin,Kelembaban,Suhu,Tek.Udara,Sol.Rad,Curah Hujan
Baris Excel,,,,,,,,,,,,,,
10,2025-01-01 00:00:00,34,24,51,1554,82,60,0,186,79,27,1007.47,0,0
11,2025-01-01 00:30:00,56,39,50,1525,81,60,0,193,81,27,1007.17,0,0
12,2025-01-01 01:00:00,45,32,50,1480,82,60,0,72,81,27,1006.84,0,0
13,2025-01-01 01:30:00,55,39,54,1672,82,60,0,0,81,27,1006.43,0,0
14,2025-01-01 02:00:00,60,42,51,1509,82,60,0,131,83,27,1005.97,0,0


,File,Sheet,Header,Satuan,Awal timestamp,Jumlah baris bertimestamp,Timestamp baris pertama,Timestamp baris terakhir,Awal nilai pengukuran (baris),Awal SO2 (baris),Rumus setelah header,Baris terpakai lain setelah header
0,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Serang Sumurpecung,8,9,10,25444,2025-01-01 00:00:00,2026-09-19 10:30:00,10,10,0,0


## Ringkasan dan batas pemeriksaan

Jumlah baris bertimestamp dihitung dari pola tanggal pada kolom Waktu. Ini bukan
jumlah nilai SO₂ yang valid. Timestamp pertama/terakhir mengikuti urutan file,
bukan minimum/maksimum waktu, dan belum membuktikan urutan atau kelengkapan seri.
Awal nilai pengukuran berarti sel tidak kosong, belum berarti lolos validasi.

In [3]:
display(pd.DataFrame(summary))
after = {str(p.relative_to(ROOT)): digest(p) for p in files}
assert before == after, 'Hash sumber berubah selama audit. Periksa file sumber.'
print('Verifikasi SHA256: seluruh file sumber tetap identik sebelum dan sesudah audit.')
print('Kendala:', issues if issues else 'Tidak ada file/sheet yang gagal dibaca.')
report = {'inventaris': inventories, 'ringkasan': summary, 'detail': details,
          'kendala': issues, 'sha256_sebelum': before, 'sha256_sesudah': after}
(OUT / 'audit_struktur_data.json').write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Laporan rinci tersimpan di outputs/audit_struktur_data.json')

Verifikasi SHA256: seluruh file sumber tetap identik sebelum dan sesudah audit.
Kendala: Tidak ada file/sheet yang gagal dibaca.
Laporan rinci tersimpan di outputs/audit_struktur_data.json


,File,Sheet,Header,Satuan,Awal timestamp,Jumlah baris bertimestamp,Timestamp baris pertama,Timestamp baris terakhir,Awal nilai pengukuran (baris),Awal SO2 (baris),Rumus setelah header,Baris terpakai lain setelah header
0,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Bandar Lampung Talang,8,9,10,28226,2025-01-01 00:00:00,2026-09-22 11:30:00,10,10,0,0
1,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Kab. Lampung Selatan Way Urang,8,9,10,29966,2025-01-01 00:00:00,2026-09-21 13:00:00,81,81,0,0
2,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Kota Cilegon Pulomerak,8,9,10,30399,2025-01-01 00:00:00,2026-09-30 17:30:00,10,10,0,0
3,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Kab. Serang Binuang,8,9,10,28732,2025-01-01 00:00:00,2026-09-30 17:30:00,10,10,0,0
4,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Kab. Tangerang Curug,8,9,10,30137,2025-01-01 00:00:00,2026-09-30 17:30:00,10,10,0,0
5,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Tangerang Pasir Jaya,8,9,10,23674,2025-01-01 00:00:00,2026-09-30 17:30:00,10,10,0,0
6,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Tangerang Selatan Serpong,8,9,10,25080,2025-01-01 00:00:00,2026-09-30 17:30:00,10,5353,0,0
7,data_raw\Telkom University Purwokerto - Intan Azizah.xlsx,Serang Sumurpecung,8,9,10,25444,2025-01-01 00:00:00,2026-09-19 10:30:00,10,10,0,0


## Hal yang perlu diperiksa pada audit berikutnya

- Kelengkapan SO₂ per stasiun dan periode: bedakan baris bertimestamp dengan nilai pengukuran tersedia.
- Tipe asli tanggal/numerik, timezone, urutan waktu, timestamp duplikat, serta interval yang hilang. Selisih 30 menit pada contoh awal belum membuktikan frekuensi seluruh seri.
- Kesesuaian rentang tanggal metadata dengan timestamp aktual setiap sheet.
- Sel kosong, nol, nilai konstan berulang, dan kode khusus. Minta dokumentasi instrumen/kode bila artinya tidak dinyatakan dalam file.
- Rentang nilai, potensi anomali dan konsistensi satuan antarstasiun; tandai dahulu tanpa mengubah nilai.
- Tentukan stasiun target, periode penelitian, frekuensi forecasting, dan horizon setelah audit kualitas selesai.

Pengamatan awal: Way Urang memiliki timestamp tetapi pengukuran kosong pada lima
baris pertama. SO2 Serpong juga kosong pada lima baris awal. Beberapa parameter
meteorologi Talang bernilai nol; SO2 Cilegon berulang pada contoh awal.
Kec.Angin Binuang di I10 tercatat 333.424 m/s. Semua ini perlu ditelusuri, tanpa
menyimpulkan penyebab atau mengubahnya pada tahap ini.